# Exemplar Partitioning — walkthrough

Load a prebuilt EP dictionary, inspect what's inside, see how the assignment routine works, and lay out an intervention recipe. CPU-only — no model load. Runs end-to-end in under a minute on a fresh machine after the first download.

## Install

From the repo root: `pip install -e .`. The notebook only needs the core install — no SAE / scripts extras required.

In [ ]:
import ep
import numpy as np

print("ep package loaded")

## Load a prebuilt dictionary

`Dictionary.from_hub` pulls from [`J-RUM/exemplar-partitioning`](https://huggingface.co/datasets/J-RUM/exemplar-partitioning). The L12 p=10 build is ~65 MB and downloads on first call; later loads are instant from the local cache.

In [ ]:
d = ep.Dictionary.from_hub("gemma-2-2b", layer=12, percentile=10)
print(d)

## Anatomy of a Dictionary

A `Dictionary` is a list of partitions plus the geometric anchors that define cell membership: `center` (the activation centroid in raw space) and `threshold` (the cosine-distance cutoff for joining a cell). Both are calibrated once from the data stream and immutable after.

In [ ]:
print(f"partitions:            {len(d.partitions)}")
print(f"center.shape:          {d.center.shape}")
print(f"||center||:            {float(np.linalg.norm(d.center)):.3f}")
print(f"threshold (cosine):    {d.threshold:.4f}")

## The largest partitions

Each partition stores the closest member prompts — the ones whose centered activation direction is most aligned with the exemplar. Reading these tells you what the cell "encodes": prompts whose L12 activation lands in that Voronoi cell.

In [ ]:
for p in sorted(d.partitions, key=lambda p: -p.member_count)[:3]:
    print(f"K={p.member_count}, coherence={p.member_coherence:.2f}")
    for dist, prompt, pos in p.closest_prompts[:3]:
        print(f"  d={dist:.3f}  pos={pos:3d}  {prompt[:80]!r}")
    print()

## The most coherent partitions

`member_coherence` ∈ [0, 1] measures how aligned the member directions are — 1 means every member points the same way. High-coherence cells encode something specific; low-coherence cells are loose. Filter out tiny cells so we're comparing like with like.

In [ ]:
populous = [p for p in d.partitions if p.member_count >= 100]
for p in sorted(populous, key=lambda p: -p.member_coherence)[:3]:
    print(f"coherence={p.member_coherence:.3f}, K={p.member_count}")
    for dist, prompt, pos in p.closest_prompts[:3]:
        print(f"  d={dist:.3f}  pos={pos:3d}  {prompt[:80]!r}")
    print()

## Assign new activations

`Dictionary.assign` returns the nearest partition for each activation and its cosine distance to that partition's exemplar. This is the same routine that runs at build time; at inference you can use it to label activations or to gate on the distance.

Nearest-exemplar distance is also the OOD signal the paper relies on: real activations from outside the build distribution land farther from any exemplar than in-distribution ones. The signal is strongest with tight dictionaries (p=1, p=2) — on the loose p=10 build loaded here, the 203 cells cover most of activation space, so even random noise lands inside a cell. To see the OOD geometry, load `percentile=2` (1.7 GB) or run [`scripts/exp_coverage.py`](../scripts/exp_coverage.py).

In [ ]:
rng = np.random.default_rng(0)
random_acts = rng.standard_normal((100, d.center.shape[0])).astype(np.float32)
partition_ids, dists = d.assign(random_acts)

print(f"assigned 100 activations to {len(set(partition_ids))} distinct partitions")
print(f"mean cosine distance to nearest exemplar: {dists.mean():.3f}")
print(f"dictionary threshold:                     {d.threshold:.3f}")

## Intervention pattern (requires a live model)

The cells above run on CPU because they only touch the dictionary. To actually steer or ablate you need a forward pass — the recipe below is rendered as a string so the notebook doesn't try to execute it.

Two things to watch:

- `alpha` controls push strength. `exemplar_direction` is unit-norm, so `alpha` directly sets how strongly you push in raw activation units. `||d.center||` is a reasonable starting point; if you see no behavioural effect, try `2·||d.center||` before concluding the partition is causally inert.
- Hook the same layer you built the dictionary at. Building at L12 and hooking at L20 will look like a null result.

In [ ]:
intervention_recipe = r"""
import torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gemma-2-2b", device="cuda")

p = d.partitions[42]
e = torch.from_numpy(p.exemplar_direction)
c = torch.from_numpy(d.center)

def steer(act, hook, alpha=float(torch.linalg.norm(c))):
    return act + alpha * e.to(act.device, act.dtype)

model.add_hook("blocks.12.hook_resid_post", steer, "fwd")
print(model.generate("The story begins", max_new_tokens=30))
"""
print(intervention_recipe)

## Where to go from here

- [`scripts/README.md`](../scripts/README.md) — full set of paper-reproduction scripts, grouped by what they produce.
- §4.1 AxBench AUROC: `python -m scripts.build_partitions --model google/gemma-2-2b-it --model-short gemma-2-2b-it --layer 20 --percentile 1 --max-tokens 10_000_000 --eval axbench`.
- Refusal collapse (§4.1 + appendix §A.2): `python -m scripts.exp_behavioral` then `python -m scripts.make_fig_refusal`.
- A different cell: change `percentile=10` → `percentile=2` (or `1`) to load a tighter dictionary with ~5k–20k partitions. The p=1 build is ~6 GB; first download is slow.